# Chapter 38: Camera Calibration & Perspective Transformation
## Multimedia & Computer Vision Learning Hub - ITERA
**Author:** Aji Pamungkas Tri Nurcahyo  
**Institution:** Institut Teknologi Sumatera (ITERA)  
**Objective:** Menguasai model kamera pinhole, matriks intrinsik K, parameter ekstrinsik [R|t], distorsi lensa Brown-Conrady, dan metode kalibrasi Zhang.

### 1. Landasan Teori & Formulasi Matematis
Model proyeksi kamera lubang jarum ideal menghubungkan titik dunia 3D $\mathbf{X}_w$ ke titik piksel 2D $\mathbf{x}$ melalui:
$$\lambda \begin{bmatrix} u \\ v \\ 1 \end{bmatrix} = \mathbf{K} [\mathbf{R} \mid \mathbf{t}] \begin{bmatrix} X_w \\ Y_w \\ Z_w \\ 1 \end{bmatrix}$$

Di mana matriks parameter intrinsik $\mathbf{K}$ didefinisikan sebagai:
$$\mathbf{K} = \begin{bmatrix} f_x & s & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}$$

Model distorsi radial lensa Brown-Conrady:
$$x_{\text{dist}} = x (1 + k_1 r^2 + k_2 r^4), \quad y_{\text{dist}} = y (1 + k_1 r^2 + k_2 r^4)$$

In [ ]:
# 2. Dependency Setup & Environment
import numpy as np
import cv2
print(f"NumPy Version: {np.__version__}")
print(f"OpenCV Version: {cv2.__version__}")

In [ ]:
# 3. Sintesis Pola Titik Sudut Papan Catur 3D (Chessboard Calibration Pattern)
# Definisikan pola papan catur 7x7 titik sudut interior dengan ukuran kotak 25 mm
pattern_size = (7, 7)
square_size = 25.0  # mm

# Koordinat dunia 3D pada bidang Z=0
objp = np.zeros((pattern_size[0] * pattern_size[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:pattern_size[0], 0:pattern_size[1]].T.reshape(-1, 2) * square_size

print(f"Total titik kalibrasi per bidang: {objp.shape[0]}")
print(f"Rentang koordinat dunia: X in [{objp[:, 0].min()}, {objp[:, 0].max()}] mm, Y in [{objp[:, 1].min()}, {objp[:, 1].max()}] mm")

In [ ]:
# 4. Simulasi Proyeksi Citra Kamera dengan Parameter Ground Truth
# Tetapkan parameter intrinsik kamera acuan
K_true = np.array([
    [600.0,   0.0, 320.0],
    [  0.0, 600.0, 240.0],
    [  0.0,   0.0,   1.0]
], dtype=np.float64)

dist_true = np.array([-0.15, 0.05, 0.001, -0.001, 0.0], dtype=np.float64)  # Barrel distortion

# Buat 5 orientasi pose kamera yang berbeda terhadap papan catur
objpoints = []
imgpoints = []

np.random.seed(42)
for view_idx in range(5):
    # Variasikan sudut rotasi (yaw, pitch, roll dalam batas +- 25 derajat)
    rvec = np.radians(np.array([np.random.uniform(-15, 15), np.random.uniform(-15, 15), np.random.uniform(-20, 20)]))
    tvec = np.array([np.random.uniform(-30, 30), np.random.uniform(-30, 30), np.random.uniform(400, 600)])
    
    pts_2d, _ = cv2.projectPoints(objp, rvec, tvec, K_true, dist_true)
    # Tambahkan sedikit derau pengukuran sub-piksel Gaussian (sigma = 0.1 px)
    noise = np.random.normal(0, 0.1, pts_2d.shape)
    pts_2d_noisy = (pts_2d + noise).astype(np.float32)
    
    objpoints.append(objp)
    imgpoints.append(pts_2d_noisy)

print(f"Berhasil menghasilkan {len(imgpoints)} sudut pandang kalibrasi sintetis.")

In [ ]:
# 5. Eksekusi Kalibrasi Kamera Menggunakan Algoritma Zhang
image_shape = (640, 480)
ret, K_est, dist_est, rvecs_est, tvecs_est = cv2.calibrateCamera(
    objpoints, imgpoints, image_shape, None, None
)

print(f"Reprojection Error Total (RMS): {ret:.4f} piksel")
print("Matriks Intrinsik K Terestimasi:\n", np.round(K_est, 2))
print("Ground Truth K:\n", K_true)
print("Koefisien Distorsi Terestimasi (k1, k2, p1, p2):\n", np.round(dist_est.ravel()[:4], 4))

# Verifikasi keakuratan estimasi panjang fokus
fx_err = np.abs(K_est[0, 0] - K_true[0, 0])
print(f"Galat Estimasi Panjang Fokus fx: {fx_err:.2f} piksel ({fx_err/K_true[0, 0]*100:.2f}%)")
assert ret < 0.5, "Reprojection error harus berada di bawah 0.5 piksel untuk kalibrasi berkualitas"

### 6. Batasan & Analisis Kegagalan (Failure Modes)
1. **Konfigurasi Degenerat**: Semua citra diambil dari orientasi paralel murni tanpa kemiringan perspektif (menyebabkan persamaan Zhang tidak konvergen).
2. **Pola Non-Datar**: Bila lembar papan catur ditempel pada media yang bergelombang atau melengkung, estimasi focal length akan melenceng jauh.
3. **Distorsi Ekstrem Fisheye**: Polinomial derajat 4-6 Brown-Conrady gagal memodelkan sudut pandang FOV > 120 derajat.

### 7. Referensi Akademik
- Zhang, Z. (2000). *A Flexible New Technique for Camera Calibration*. IEEE TPAMI.
- Brown, D. C. (1971). *Close-range camera calibration*. Photogrammetric Engineering.